# VASP: Otimização de Geometria

Autor: [Prof. Elvis do A. Soares](https://github.com/elvissoares) 

Contato: [elvis@peq.coppe.ufrj.br](mailto:elvis@peq.coppe.ufrj.br) - [Programa de Engenharia Química, PEQ/COPPE, UFRJ, Brasil](https://www.peq.coppe.ufrj.br/)

---

## Otimizando a geometria da molécula de H2O

Criando a molécula de água (H2O) utilizando o ASE

In [17]:
from ase import Atoms, Atom

h2omol = Atoms([Atom('O', [0, 0, 0]),
                Atom('H', [0.0, -0.760265, 0.588373]),
                Atom('H', [0.0, 0.760265, 0.588373])])

h2omol.center(vacuum=4.0) # caixa com 4 Angstroms de vácuo 
h2omol.pbc = True # condição de contorno periódica

Visualizando a molécula com o ASE

In [18]:
from ase.visualize import view

view(h2omol, viewer='x3d')

Importando o VASP calculator para o ASE e anexando a calculadora a molécula

In [19]:
from ase.calculators.vasp import Vasp

calc = Vasp(directory='H2O_relaxed',
            xc='PBE',   # funcional GGA
            encut=350,  # safe default for PAW-PBE sets
            kpts=[1, 1, 1],gamma=True,                  # k-points
            ibrion=2, # CG ionic relax
            isif=0, # relaxa somente os átomos, mantendo a célula fixa
            nsw=50, # número máximo de passos de relaxação
            ediffg=-1e-3, # critério de convergência para relaxação (forças)
            lreal='Auto', # projeção de orbitais no espaço real
            lwave=True, lcharg=True,lvtot=True,  # mantem WAVECAR/CHGCAR/LOCPOT para post-processing
            )

h2omol.calc = calc

Exportando resultado da energia

In [20]:
E_h2o = h2omol.get_potential_energy()       

print(f'Energia total da molécula de água: {E_h2o:.3f} eV')

Energia total da molécula de água: -14.221 eV


Quantos passos iônicos foram feitos

In [21]:
print("Número de passos iônicos executados:", calc.get_number_of_iterations())

Número de passos iônicos executados: 2


Quais as forças sobre os átomos

In [22]:
forces = h2omol.get_forces()
print("Forças sobre os átomos (eV/Ang):")
print(forces)

Forças sobre os átomos (eV/Ang):
[[ 0.          0.          0.00071941]
 [-0.          0.00046482 -0.0003597 ]
 [-0.         -0.00046482 -0.0003597 ]]


A nova geometria da molécula

In [23]:
print("Nova geometria da molécula de água (Ang):")
print(h2omol.get_positions())

Nova geometria da molécula de água (Ang):
[[4.         4.760265   3.99497349]
 [4.         3.99123747 4.59088625]
 [4.         5.52929253 4.59088625]]


Calculando os comprimentos de ligação OH e o ângulo de ligação HOH

In [24]:
lOH1 = h2omol.get_distance(0, 1)
lOH2 = h2omol.get_distance(0, 2)
thetaOH1H2 = h2omol.get_angle(1, 0, 2)

print(f"O comprimento da ligação O-H1 é {lOH1:.3f} Å")
print(f"O comprimento da ligação O-H2 é {lOH2:.3f} Å")
print(f"O ângulo H1-O-H2 é {thetaOH1H2:.2f}°")

O comprimento da ligação O-H1 é 0.973 Å
O comprimento da ligação O-H2 é 0.973 Å
O ângulo H1-O-H2 é 104.46°


## Otimizando geometria do Cristal de NaCl

Criando cristal de NaCl

In [25]:
from ase.build import bulk

Nacl_crystal = bulk("NaCl", crystalstructure="rocksalt", a=5.64, cubic=True)

print(Nacl_crystal)
print("Cell:", Nacl_crystal.get_cell())
print("Positions:\n", Nacl_crystal.get_positions())

Atoms(symbols='NaClNaClNaClNaCl', pbc=True, cell=[5.64, 5.64, 5.64])
Cell: Cell([5.64, 5.64, 5.64])
Positions:
 [[0.   0.   0.  ]
 [2.82 0.   0.  ]
 [0.   2.82 2.82]
 [2.82 2.82 2.82]
 [2.82 0.   2.82]
 [0.   0.   2.82]
 [2.82 2.82 0.  ]
 [0.   2.82 0.  ]]


visualizando estrutura

In [26]:
view(Nacl_crystal, viewer='x3d')

Criando calculadora do VASP e anexando ao sistema cristalino

In [ ]:
calc = Vasp(directory='NaCl_relaxed',
            xc='PBE',   # funcional GGA
            encut=350,  # safe default for PAW-PBE sets
            kpts=[1, 1, 1],gamma=True,                  # k-points
            ibrion=2, # CG ionic relax
            isif=7, # relaxa somente a célula
            nsw=50, # número máximo de passos de relaxação
            ediffg=-1e-3, # critério de convergência para relaxação (forças)
            lreal='Auto', # projeção de orbitais no espaço real
            lwave=True, lcharg=True,lvtot=True,  # mantem WAVECAR/CHGCAR/LOCPOT para post-processing
)

Nacl_crystal.calc = calc

Energia da célula unitária primitiva

In [36]:
E_nacl = Nacl_crystal.get_potential_energy()
print(f'Energia total do cristal de NaCl: {E_nacl:.3f} eV')

Energia total do cristal de NaCl: -25.451 eV


Quantos passos iônicos?

In [39]:
print("Número de passos de otimização da célula executados:", calc.get_number_of_iterations())

Número de passos de otimização da célula executados: 4


Qual a nova estrutura?

In [37]:
print("Nova estrutura do cristal de NaCl (Ang):")
print(Nacl_crystal.get_cell())

Nova estrutura do cristal de NaCl (Ang):
Cell([5.911942143216201, 5.911942143216201, 5.911942143216201])
